<a href="https://colab.research.google.com/github/cibelerusso/EstatisticaI/blob/main/Notebooks/Probabilidade_Parte_III_Exemplos_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SME0123 - Estatística - Parte III: Probabilidade

## Profa. Cibele Russo (ICMC USP)

## Notebook Python dos Exemplos 3.1 a 3.15

Criado com apoio da ferramenta de IA generativa ChatGPT (Open AI).

Este notebook reproduz computacionalmente os **exemplos enumerados** do material, usando principalmente `scipy.stats`.

Observações de parametrização importantes:

- `scipy.stats.geom(p)` conta o **número de ensaios até o primeiro sucesso** (suporte 1, 2, ...). Quando o material conta **fracassos antes do primeiro sucesso**, usamos a equivalência `X = Y - 1` ou `nbinom(n=1, p)`.
- `scipy.stats.nbinom(r, p)` conta o **número de fracassos antes do r-ésimo sucesso**, exatamente como no material.
- `scipy.stats.norm(loc=mu, scale=sigma)` usa **desvio-padrão** em `scale`, e não variância.

In [1]:
import math
import numpy as np
from scipy.stats import randint, bernoulli, binom, nbinom, hypergeom, poisson, uniform, expon, norm

np.set_printoptions(precision=6, suppress=True)

## Exemplo 3.1 - Distribuição Uniforme Discreta

Seja $X$ o número de pontos na face superior de um dado perfeito. Então
$X\in\{1,2,3,4,5,6\}$ e $P(X=x)=1/6$.

Objetivo: obter a f.m.p., a esperança e a variância.

In [2]:
# Dado perfeito: uniforme discreta em {1, 2, ..., 6}
X = randint(low=1, high=7)  # high é exclusivo
x = np.arange(1, 7)
pmf = X.pmf(x)

print("x        =", x)
print("P(X=x)   =", pmf)
print("E(X)     =", X.mean())
print("Var(X)   =", X.var())
print("35/12    =", 35/12)

x        = [1 2 3 4 5 6]
P(X=x)   = [0.166667 0.166667 0.166667 0.166667 0.166667 0.166667]
E(X)     = 3.5
Var(X)   = 2.9166666666666665
35/12    = 2.9166666666666665


## Exemplo 3.2 - Distribuição de Bernoulli

Lança-se um dado perfeito e define-se sucesso como “ocorrer a face 3”. Logo,
$p=1/6$ e $X\sim Ber(1/6)$, com $X=1$ em caso de sucesso e $X=0$ caso contrário.

In [3]:
p = 1/6
X = bernoulli(p)

x = np.array([0, 1])
print("x        =", x)
print("P(X=x)   =", X.pmf(x))
print("E(X)     =", X.mean())
print("Var(X)   =", X.var())

x        = [0 1]
P(X=x)   = [0.833333 0.166667]
E(X)     = 0.16666666666666666
Var(X)   = 0.1388888888888889


## Exemplo 3.3 - Distribuição Binomial (teórico)

Três peças são classificadas como não defeituosas ($X_i=1$) ou defeituosas ($X_i=0$),
com $P(X_i=1)=p$. Para $Y=X_1+X_2+X_3$, tem-se

$$Y\sim Bin(3,p),\qquad P(Y=y)=\binom{3}{y}p^y(1-p)^{3-y},\ y=0,1,2,3.$$

O código abaixo mantém $p$ como parâmetro editável.

In [4]:
from math import comb

def pmf_exemplo_33(p):
    y = np.arange(0, 4)
    probs = np.array([comb(3, k) * p**k * (1-p)**(3-k) for k in y])
    return y, probs

# Fórmulas dos quatro valores possíveis
for y in range(4):
    print(f"P(Y={y}) = C(3,{y}) * p^{y} * (1-p)^{3-y}")

# Ilustração numérica: altere p se desejar
p = 0.8
y, probs = pmf_exemplo_33(p)
print()
print("Ilustração com p =", p)
print("y        =", y)
print("P(Y=y)   =", probs)
print("soma     =", probs.sum())
print("E(Y)     =", 3*p)
print("Var(Y)   =", 3*p*(1-p))

P(Y=0) = C(3,0) * p^0 * (1-p)^3
P(Y=1) = C(3,1) * p^1 * (1-p)^2
P(Y=2) = C(3,2) * p^2 * (1-p)^1
P(Y=3) = C(3,3) * p^3 * (1-p)^0

Ilustração com p = 0.8
y        = [0 1 2 3]
P(Y=y)   = [0.008 0.096 0.384 0.512]
soma     = 1.0
E(Y)     = 2.4000000000000004
Var(Y)   = 0.48


## Exemplo 3.4 - Binomial em dois níveis

A taxa de equipamentos em perfeito estado é 97%.

**(a)** Em 20 itens, calcular a probabilidade de haver pelo menos um defeituoso.

**(b)** Em 10 carregamentos, cada um com 20 itens, calcular a probabilidade de exatamente 3 carregamentos apresentarem pelo menos um defeituoso.

In [5]:
p_bom = 0.97
n_itens = 20

# (a) Complementar: pelo menos um defeituoso = 1 - todos em perfeito estado
q = 1 - p_bom**n_itens
print(f"(a) P(pelo menos 1 defeituoso em 20) = {q:.10f}")

# (b) Cada carregamento é um Bernoulli com 'sucesso' = ter pelo menos 1 defeituoso
n_carregamentos = 10
prob_3 = binom.pmf(3, n_carregamentos, q)
print(f"(b) P(exatamente 3 carregamentos com defeito) = {prob_3:.10f}")

(a) P(pelo menos 1 defeituoso em 20) = 0.4562056571
(b) P(exatamente 3 carregamentos com defeito) = 0.1602160581


## Exemplo 3.5 - Distribuição Geométrica

Cada reação é positiva com probabilidade $p=0{,}3$. Seja $X$ o número de reações negativas antes da primeira positiva.
Queremos $P(X<3)=P(X\le 2)$.

Como `nbinom(1,p)` conta fracassos antes do primeiro sucesso, ela coincide com esta parametrização geométrica.

In [6]:
p = 0.3
# X = número de fracassos antes do primeiro sucesso
prob = nbinom.cdf(2, n=1, p=p)

# Equivalente via soma direta: x = 0,1,2
prob_manual = sum(p * (1-p)**x for x in range(3))

print(f"P(X < 3) = {prob:.10f}")
print(f"Checagem por soma direta = {prob_manual:.10f}")

P(X < 3) = 0.6570000000
Checagem por soma direta = 0.6570000000


## Exemplo 3.6 - Distribuição Binomial Negativa

O time A vence cada jogo com probabilidade $p=0{,}55$ e vence a série ao obter a 4ª vitória.
“Vencer a série em seis jogos” significa que a 4ª vitória ocorre no 6º jogo, ou seja, há exatamente 2 derrotas antes da 4ª vitória.

Logo, se $X$ é o número de fracassos antes do 4º sucesso, queremos $P(X=2)$.

In [7]:
p = 0.55
r = 4
fracassos = 2

prob_nbinom = nbinom.pmf(fracassos, n=r, p=p)
prob_manual = math.comb(5, 3) * p**4 * (1-p)**2

print(f"P(A vencer em exatamente 6 jogos) = {prob_nbinom:.10f}")
print(f"Checagem combinatória = {prob_manual:.10f}")

P(A vencer em exatamente 6 jogos) = 0.1853001562
Checagem combinatória = 0.1853001563


## Exemplo 3.7 - Distribuição Hipergeométrica

Em um lote de 20 peças, 4 são defeituosas. Selecionam-se 5 peças sem reposição.
Queremos a probabilidade de selecionar exatamente 2 defeituosas.

Na parametrização do SciPy: `hypergeom(M, n, N)`, em que
`M=20` é o tamanho da população, `n=4` o número de defeituosas e `N=5` o tamanho da amostra.

In [8]:
M = 20   # população
n = 4    # defeituosas na população
N = 5    # tamanho da amostra
x = 2

prob = hypergeom.pmf(x, M=M, n=n, N=N)
prob_manual = math.comb(4, 2) * math.comb(16, 3) / math.comb(20, 5)

print(f"P(X=2) = {prob:.10f}")
print(f"Checagem combinatória = {prob_manual:.10f}")

P(X=2) = 0.2167182663
Checagem combinatória = 0.2167182663


## Exemplo 3.8 - Distribuição de Poisson

Uma central recebe, em média, 5 chamadas por minuto. Para um intervalo de 1 minuto,
$X\sim Poi(5)$. Queremos $P(X\le 2)$.

In [9]:
lamb = 5
prob = poisson.cdf(2, mu=lamb)

print(f"P(X <= 2), X ~ Poi(5) = {prob:.10f}")

P(X <= 2), X ~ Poi(5) = 0.1246520195


## Exemplo 3.9 - Processo de Poisson

Mantendo a taxa de 5 chamadas por minuto, em 4 minutos a média é
$\mu=\lambda t=5\times 4=20$. Portanto $X\sim Poi(20)$ e queremos $P(X\le 2)$.

In [10]:
taxa = 5
minutos = 4
mu = taxa * minutos
prob = poisson.cdf(2, mu=mu)

print("Média em 4 minutos =", mu)
print(f"P(X <= 2), X ~ Poi(20) = {prob:.10f}")

Média em 4 minutos = 20
P(X <= 2), X ~ Poi(20) = 0.0000004555


## Exemplo 3.10 - Aproximação da Binomial pela Poisson

A probabilidade de acidente em um dia é $p=0{,}005$, durante $M=400$ dias independentes.
O número de dias com acidente teria distribuição binomial $Bin(400,0{,}005)$.
Como $M$ é grande e $p$ é pequeno, o material usa a aproximação de Poisson com
$\lambda=Mp=2$.

Queremos a probabilidade de no máximo 3 dias com acidente.

In [11]:
M = 400
p = 0.005
lamb = M * p

prob_aprox = poisson.cdf(3, mu=lamb)
print("lambda = M*p =", lamb)
print(f"Aproximação de Poisson: P(X <= 3) = {prob_aprox:.10f}")

lambda = M*p = 2.0
Aproximação de Poisson: P(X <= 3) = 0.8571234605


## Exemplo 3.11 - Distribuição Uniforme Contínua

Seja $X\sim U(-1/2,1/2)$.

**(a)** Determinar $F_X(x)$.

**(b)** Calcular $P(-1/4<X\le 1/4)$.

**(c)** Calcular $E(X)$ e $Var(X)$.

In [12]:
a = -1/2
b = 1/2
X = uniform(loc=a, scale=b-a)

# (a) Função distribuição acumulada (implementação explícita)
def F_uniforme(x, a=-0.5, b=0.5):
    x = np.asarray(x, dtype=float)
    return np.where(x < a, 0.0,
                    np.where(x <= b, (x-a)/(b-a), 1.0))

pontos = np.array([-1.0, -0.5, -0.25, 0.0, 0.25, 0.5, 1.0])
print("(a) x       =", pontos)
print("    F_X(x)  =", F_uniforme(pontos))

# (b)
prob = X.cdf(1/4) - X.cdf(-1/4)
print(f"(b) P(-1/4 < X <= 1/4) = {prob:.10f}")

# (c)
print(f"(c) E(X)   = {X.mean():.10f}")
print(f"    Var(X) = {X.var():.10f}")
print(f"    1/12   = {1/12:.10f}")

(a) x       = [-1.   -0.5  -0.25  0.    0.25  0.5   1.  ]
    F_X(x)  = [0.   0.   0.25 0.5  0.75 1.   1.  ]
(b) P(-1/4 < X <= 1/4) = 0.5000000000
(c) E(X)   = 0.0000000000
    Var(X) = 0.0833333333
    1/12   = 0.0833333333


## Exemplo 3.12 - Distribuição Exponencial

O tempo de vida $X$ de um fusível tem distribuição exponencial com média de 100 horas.
Para a exponencial, $E(X)=1/\lambda$, portanto $\lambda=1/100$ por hora.
Queremos $P(X>150)$.

In [13]:
media = 100
lamb = 1 / media
X = expon(scale=media)  # scale = 1/lambda = média

prob = X.sf(150)  # survival function = P(X > 150)
print("lambda =", lamb)
print(f"P(X > 150) = {prob:.10f}")
print(f"Checagem exp(-lambda*150) = {np.exp(-lamb*150):.10f}")

lambda = 0.01
P(X > 150) = 0.2231301601
Checagem exp(-lambda*150) = 0.2231301601


## Exemplo 3.13 - Normal-padrão

Seja $Z\sim N(0,1)$. Calcular:

- (a) $P(0\le Z\le 1{,}65)$
- (b) $P(Z\le 0{,}5)$
- (c) $P(Z\le -1{,}57)$
- (d) $P(-0{,}65\le Z\le 0{,}65)$
- (e) $P(0{,}80\le Z\le 1{,}40)$
- (f) $z$ tal que $P(0\le Z\le z)=0{,}4753$
- (g) $z$ tal que $P(Z\le z)=0{,}05$

In [14]:
Z = norm(loc=0, scale=1)

pa = Z.cdf(1.65) - Z.cdf(0)
pb = Z.cdf(0.5)
pc = Z.cdf(-1.57)
pd = Z.cdf(0.65) - Z.cdf(-0.65)
pe = Z.cdf(1.40) - Z.cdf(0.80)

# (f): Phi(z) - 0.5 = 0.4753  => Phi(z) = 0.9753
zf = Z.ppf(0.5 + 0.4753)

# (g): Phi(z) = 0.05
zg = Z.ppf(0.05)

print(f"(a) {pa:.6f}")
print(f"(b) {pb:.6f}")
print(f"(c) {pc:.6f}")
print(f"(d) {pd:.6f}")
print(f"(e) {pe:.6f}")
print(f"(f) z = {zf:.6f}")
print(f"(g) z = {zg:.6f}")

(a) 0.450529
(b) 0.691462
(c) 0.058208
(d) 0.484308
(e) 0.131099
(f) z = 1.965123
(g) z = -1.644854


## Exemplo 3.14 - Distribuição Normal

Seja $X\sim N(90,100)$. Na notação do material, o segundo parâmetro é a variância,
logo $\mu=90$, $\sigma^2=100$ e $\sigma=10$.

Calcular:

- (a) $P(80\le X\le 100)$
- (b) $P(X\le 90)$
- (c) $P(60\le X\le 75)$

In [15]:
mu = 90
variancia = 100
sigma = np.sqrt(variancia)
X = norm(loc=mu, scale=sigma)

pa = X.cdf(100) - X.cdf(80)
pb = X.cdf(90)
pc = X.cdf(75) - X.cdf(60)

print(f"(a) P(80 <= X <= 100) = {pa:.10f}")
print(f"(b) P(X <= 90)        = {pb:.10f}")
print(f"(c) P(60 <= X <= 75)  = {pc:.10f}")

(a) P(80 <= X <= 100) = 0.6826894921
(b) P(X <= 90)        = 0.5000000000
(c) P(60 <= X <= 75)  = 0.0654573032


## Exemplo 3.15 - Aproximação da Binomial pela Normal

Uma prova tem 20 questões, cada uma com 4 alternativas. Respondendo ao acaso,
a probabilidade de acerto é $p=1/4$. Assim, o número de acertos é
$X\sim Bin(20,0{,}25)$.

O material introduz a aproximação normal $Y\sim N(Mp, Mp(1-p))$ e, sem correção de continuidade,
usa $P(X\ge a)\approx P(Y\ge a)$.

“Acertar 50% ou mais” corresponde a $X\ge 10$.

In [16]:
M = 20
p = 0.25
mu = M * p
var = M * p * (1-p)
sigma = np.sqrt(var)

Y = norm(loc=mu, scale=sigma)
prob_aprox = Y.sf(10)  # P(Y >= 10) = P(Y > 10) para variável contínua

print("mu       =", mu)
print("variância=", var)
print("sigma    =", sigma)
print(f"Aproximação normal sem correção de continuidade: {prob_aprox:.10f}")

mu       = 5.0
variância= 3.75
sigma    = 1.9364916731037085
Aproximação normal sem correção de continuidade: 0.0049116373


## Resumo das distribuições usadas

| Exemplo | Modelo principal |
|---|---|
| 3.1 | Uniforme discreta |
| 3.2 | Bernoulli |
| 3.3–3.4 | Binomial |
| 3.5 | Geométrica |
| 3.6 | Binomial negativa |
| 3.7 | Hipergeométrica |
| 3.8–3.10 | Poisson / aproximação de Poisson |
| 3.11 | Uniforme contínua |
| 3.12 | Exponencial |
| 3.13–3.14 | Normal |
| 3.15 | Aproximação normal da Binomial |